# SignalScope - Ensemble Combiner
Runs each frozen member over a labelled sample, then fits a logistic-regression
combiner on their output probabilities.

The combiner is fitted on **seen generators + vq_diffusion** and evaluated on
**stable_diffusion + glide**, which no member and no fit ever touches.

In [ ]:
import os
import json
import glob
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
import torchvision.transforms as T
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

# Keep inference tractable: three transformer members over every image is the
# expensive part, not the fitting.
N_FIT = 4000
N_UNSEEN = 4000
BATCH = 32

## 1. Rebuild the exact same splits
Identical filtering and holdout rules to the classifier run, so the unseen
numbers are directly comparable.

In [ ]:
ARTIFACT_ROOT = "/kaggle/input/datasets/awsaf49/artifact-dataset"

folders = {}
for name in sorted(os.listdir(ARTIFACT_ROOT)):
    d = os.path.join(ARTIFACT_ROOT, name)
    if os.path.isdir(d) and os.path.exists(os.path.join(d, "metadata.csv")):
        folders[name] = d
print(f"found {len(folders)} source folders")

FACE_EXCLUDE = {
    "ffhq", "celebahq", "metfaces", "face_synthetics", "sfhq",
    "stylegan1", "stylegan2", "stylegan3", "star_gan", "mat",
}
FACE_PATH_TOKENS = ("face", "ffhq", "celeba")
UNSEEN_GENERATORS = ["stable_diffusion", "glide"]
DEV_GENERATOR = "vq_diffusion"

rows = []
for name, path in sorted(folders.items()):
    if name.lower() in FACE_EXCLUDE:
        continue
    meta = pd.read_csv(os.path.join(path, "metadata.csv"))[["image_path", "target"]].copy()
    meta = meta[~meta["image_path"].str.lower().str.contains("|".join(FACE_PATH_TOKENS), regex=True, na=False)]
    meta["source"] = name
    meta["label"] = (meta["target"].astype(int) != 0).astype(int)
    meta["abspath"] = path.rstrip("/") + "/" + meta["image_path"].astype(str)
    rows.append(meta[["abspath", "label", "source"]])

catalog = pd.concat(rows, ignore_index=True)
print("total usable images:", len(catalog))

unseen = catalog[catalog["source"].isin(UNSEEN_GENERATORS)]
dev = catalog[catalog["source"] == DEV_GENERATOR]
seen = catalog[~catalog["source"].isin(UNSEEN_GENERATORS + [DEV_GENERATOR])]

real_all = seen[seen["label"] == 0]
fake_seen = seen[seen["label"] == 1]


def balanced(fake_rows, real_rows, n):
    half = min(n // 2, len(fake_rows), len(real_rows))
    return pd.concat([
        fake_rows.sample(half, random_state=SEED),
        real_rows.sample(half, random_state=SEED),
    ], ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)


# Reals are partitioned so the fit and unseen sets never share an image.
real_shuffled = real_all.sample(frac=1.0, random_state=SEED)
real_fit, real_unseen = real_shuffled.iloc[: len(real_shuffled) // 2], real_shuffled.iloc[len(real_shuffled) // 2:]

fit_fakes = pd.concat([fake_seen.sample(min(len(fake_seen), N_FIT), random_state=SEED),
                       dev[dev["label"] == 1]], ignore_index=True)
fit_df = balanced(fit_fakes, real_fit, N_FIT)
unseen_df = balanced(unseen[unseen["label"] == 1], real_unseen, N_UNSEEN)

print(f"fit set={len(fit_df)} (fake {fit_df.label.sum()})   unseen set={len(unseen_df)} (fake {unseen_df.label.sum()})")

## 2. Frozen members
Three pretrained HuggingFace detectors plus the dual-stream checkpoint.
Nothing here is trained.

In [ ]:
class FrequencyBranch(nn.Module):
    def __init__(self, in_channels=1, feature_dim=128):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 32, 3, stride=2, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, stride=2, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, stride=2, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))
        self.fc = nn.Linear(128 * 4 * 4, feature_dim)

    def extract_fft_spectrum(self, x):
        gray = 0.2989 * x[:, 0:1] + 0.5870 * x[:, 1:2] + 0.1140 * x[:, 2:3] if x.shape[1] == 3 else x
        log_spectrum = torch.log(torch.abs(torch.fft.fftshift(torch.fft.fft2(gray))) + 1e-8)
        flat = log_spectrum.view(log_spectrum.size(0), -1)
        mn = flat.min(dim=1, keepdim=True)[0].unsqueeze(-1).unsqueeze(-1)
        mx = flat.max(dim=1, keepdim=True)[0].unsqueeze(-1).unsqueeze(-1)
        return (log_spectrum - mn) / (mx - mn + 1e-8)

    def forward(self, x):
        f = self.extract_fft_spectrum(x)
        f = F.relu(self.bn1(self.conv1(f)))
        f = F.relu(self.bn2(self.conv2(f)))
        f = F.relu(self.bn3(self.conv3(f)))
        return F.relu(self.fc(torch.flatten(self.adaptive_pool(f), 1)))


class SignalScopeDualStreamModel(nn.Module):
    def __init__(self, spatial_backbone="resnet34", pretrained=False, dropout_rate=0.3):
        super().__init__()
        base = tv_models.resnet34(weights=None)
        num_spatial = base.fc.in_features
        base.fc = nn.Identity()
        self.spatial_stream = base
        self.freq_dim = 128
        self.frequency_stream = FrequencyBranch(1, self.freq_dim)
        self.classifier = nn.Sequential(
            nn.Linear(num_spatial + self.freq_dim, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Dropout(dropout_rate), nn.Linear(256, 64), nn.ReLU(),
            nn.Dropout(dropout_rate / 2), nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.classifier(torch.cat((self.spatial_stream(x), self.frequency_stream(x)), dim=1))


ckpt_paths = glob.glob("/kaggle/input/**/best_model.pt", recursive=True)
if not ckpt_paths:
    raise SystemExit("best_model.pt not found - add the training notebook's output as a data source.")
print("dual-stream checkpoint:", ckpt_paths[0])

ckpt = torch.load(ckpt_paths[0], map_location=DEVICE, weights_only=False)
dual = SignalScopeDualStreamModel().to(DEVICE)
dual.load_state_dict(ckpt["model_state_dict"])
dual.eval()
DUAL_TEMPERATURE = float(ckpt.get("temperature", 1.0))

NORM = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
if "image_size" in ckpt:
    dual_tf = T.Compose([T.Resize(ckpt.get("native_size", 200)), T.CenterCrop(ckpt["image_size"]), T.ToTensor(), NORM])
else:
    dual_tf = T.Compose([T.Resize((224, 224)), T.ToTensor(), NORM])
print("dual-stream transform:", dual_tf)

In [ ]:
from transformers import AutoImageProcessor, AutoModelForImageClassification

HF_MEMBERS = [
    "dima806/deepfake_vs_real_image_detection",
    "umm-maybe/AI-image-detector",
    "Organika/sdxl-detector",
]

AI_TOKENS = ("fake", "artificial", "synthetic", "generated", "ai-generated", "ai_generated")


def ai_class_index(model):
    for idx, label in model.config.id2label.items():
        if any(tok in label.lower() for tok in AI_TOKENS):
            return int(idx)
    return 1


hf_models = {}
for name in HF_MEMBERS:
    proc = AutoImageProcessor.from_pretrained(name)
    mdl = AutoModelForImageClassification.from_pretrained(name).to(DEVICE).eval()
    idx = ai_class_index(mdl)
    print(f"{name}: id2label={mdl.config.id2label} -> AI index {idx}")
    hf_models[name] = (proc, mdl, idx)

## 3. Member inference
Forward passes only - no gradients, no weight updates.

In [ ]:
@torch.no_grad()
def hf_probs(name, df):
    proc, mdl, idx = hf_models[name]
    out = []
    paths = df["abspath"].tolist()
    for start in range(0, len(paths), BATCH):
        imgs = [Image.open(p).convert("RGB") for p in paths[start:start + BATCH]]
        batch = proc(images=imgs, return_tensors="pt").to(DEVICE)
        logits = mdl(**batch).logits
        out.append(torch.softmax(logits, dim=1)[:, idx].float().cpu().numpy())
        if start % (BATCH * 20) == 0:
            print(f"   {name} {start}/{len(paths)}")
    return np.concatenate(out)


@torch.no_grad()
def dual_probs(df):
    out = []
    paths = df["abspath"].tolist()
    for start in range(0, len(paths), BATCH):
        imgs = torch.stack([dual_tf(Image.open(p).convert("RGB")) for p in paths[start:start + BATCH]]).to(DEVICE)
        logits = dual(imgs) / DUAL_TEMPERATURE
        out.append(torch.sigmoid(logits).squeeze(1).float().cpu().numpy())
    return np.concatenate(out)


def member_matrix(df, tag):
    print(f"\n--- scoring {tag} ({len(df)} images) ---")
    cols, names = [], []
    for name in HF_MEMBERS:
        cols.append(hf_probs(name, df))
        names.append(name)
    cols.append(dual_probs(df))
    names.append("signalscope_dual_stream")
    return np.column_stack(cols), names


X_fit, MEMBER_NAMES = member_matrix(fit_df, "fit set")
y_fit = fit_df["label"].values

X_unseen, _ = member_matrix(unseen_df, "unseen set")
y_unseen = unseen_df["label"].values

# A member whose labels are mapped backwards shows AUC < 0.5; flip it rather
# than letting the combiner fight the inverted signal.
for j, name in enumerate(MEMBER_NAMES):
    auc = roc_auc_score(y_fit, X_fit[:, j])
    if auc < 0.5:
        print(f"flipping inverted member {name} (fit AUC {auc:.3f})")
        X_fit[:, j] = 1.0 - X_fit[:, j]
        X_unseen[:, j] = 1.0 - X_unseen[:, j]

## 4. Fit the combiner and compare

In [ ]:
def report(name, y, probs, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    cm = confusion_matrix(y, preds)
    tn, fp, fn, tp = cm.ravel()
    m = {
        "auc": float(roc_auc_score(y, probs)),
        "macro_f1": float(f1_score(y, preds, average="macro")),
        "accuracy": float((tp + tn) / cm.sum()),
        "fpr": float(fp / (fp + tn)) if (fp + tn) else 0.0,
        "confusion": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    }
    print(f"{name:38s} AUC={m['auc']:.4f}  F1={m['macro_f1']:.4f}  acc={m['accuracy']*100:.1f}%  FPR={m['fpr']*100:.1f}%")
    return m


print("\n=== individual members on the UNSEEN split ===")
member_metrics = {n: report(n, y_unseen, X_unseen[:, j]) for j, n in enumerate(MEMBER_NAMES)}

combiner = LogisticRegression(max_iter=1000, C=1.0)
combiner.fit(X_fit, y_fit)

p_unseen = combiner.predict_proba(X_unseen)[:, 1]
p_fit = combiner.predict_proba(X_fit)[:, 1]

print("\n=== ensemble ===")
m_fit = report("learned ensemble (fit set)", y_fit, p_fit)
m_unseen = report("learned ensemble (UNSEEN split)", y_unseen, p_unseen)

# Current hardcoded soft-voting weights, for an honest side-by-side.
HARDCODED = np.array([0.25, 0.22, 0.20, 0.18])
hard_w = HARDCODED[: len(MEMBER_NAMES)] / HARDCODED[: len(MEMBER_NAMES)].sum()
m_hard = report("hardcoded soft-vote (UNSEEN split)", y_unseen, X_unseen @ hard_w)

LOW_FPR_THRESHOLD = float(np.quantile(p_fit[y_fit == 0], 0.95))
print(f"\nthreshold for ~5% FPR on fit set: {LOW_FPR_THRESHOLD:.4f}")
m_unseen_lowfpr = report("learned ensemble @ 5% FPR", y_unseen, p_unseen, LOW_FPR_THRESHOLD)

print("\nlearned weights:")
for n, w in zip(MEMBER_NAMES, combiner.coef_[0]):
    print(f"  {n:40s} {w:+.4f}")
print(f"  {'intercept':40s} {combiner.intercept_[0]:+.4f}")

## 5. Save

In [ ]:
ensemble_spec = {
    "members": MEMBER_NAMES,
    "coefficients": combiner.coef_[0].tolist(),
    "intercept": float(combiner.intercept_[0]),
    "low_fpr_threshold": LOW_FPR_THRESHOLD,
    "dual_stream_temperature": DUAL_TEMPERATURE,
    "fit_size": int(len(fit_df)),
    "unseen_size": int(len(unseen_df)),
    "held_out_generators": UNSEEN_GENERATORS,
    "fit_generators_include_dev": DEV_GENERATOR,
    "metrics": {
        "individual_members_unseen": member_metrics,
        "ensemble_fit": m_fit,
        "ensemble_unseen": m_unseen,
        "ensemble_unseen_at_5pct_fpr": m_unseen_lowfpr,
        "hardcoded_softvote_unseen": m_hard,
    },
}

with open("/kaggle/working/ensemble_weights.json", "w") as f:
    json.dump(ensemble_spec, f, indent=2)

print("\nSaved /kaggle/working/ensemble_weights.json")